# NLP - DistilBERT multilingual + XGboost

- Nivel 2: Few-Shot - Aqui, além do texto "embeddado", fornecemos ao modelo alguns exemplos de tickets já classificados, dentro da propria base. 


detalhe importante: 
    Embora esta fase seja chamada "few-shot", usamos treino 80/20
    para aproveitar melhor os dados disponíveis e obter métricas mais confiáveis,
    mantendo a ideia de aprendizado a partir de exemplos representativos. 
    O fel-shot comum seria com 3 4 exemplos de cada classe, ideal para fazer provas de conceito iniciais, quando s etem poucos dados reais. 


Outro detalhe importante é que o DistilBERT variante via SentenceTransformer, foi usado apenas para o NLP, no sentido de criação de embeddings, ou seja feature extraction usando NLP. Depois usado XGboost para classificar. 


In [1]:
# Imports
# ------------------------------
#import sys
#!{sys.executable} -m pip install tqdm
#!pip install numpy==1.25.2
#!pip uninstall torch -y
#!pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu118

import pandas as pd
import numpy as np
import warnings
from sentence_transformers import SentenceTransformer
from sklearn.preprocessing import LabelEncoder, label_binarize
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score, log_loss, brier_score_loss
from xgboost import XGBClassifier
import optuna
from tqdm import tqdm


In [2]:
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', None)
warnings.filterwarnings('ignore')

In [3]:
#arquivo pqt ja tratado (gerado no ETL_helpdesk_glpi_ifpr.ipynb)
arquivo_parquet = "ifpr_helpdesk_completo.parquet"
df_filtrado = pd.read_parquet(arquivo_parquet)

df_filtrado.head()


,Grupo,Categoria,Satisfação,Usuário Iteração,Resolvido,texto,Data_Abertura_Ano,Data_Abertura_Mes,Data_Abertura_Dia,Data_Abertura_DiaSemana,Data_Abertura_Hora,Data_Abertura_FimDeSemana,Data_Abertura_Periodo,Data_Solução_Ano,Data_Solução_Mes,Data_Solução_Dia,Data_Solução_DiaSemana,Data_Solução_Hora,Data_Solução_FimDeSemana,Data_Solução_Periodo,Data_Fechamento_Ano,Data_Fechamento_Mes,Data_Fechamento_Dia,Data_Fechamento_DiaSemana,Data_Fechamento_Hora,Data_Fechamento_FimDeSemana,Data_Fechamento_Periodo,Data_Iteração_Ano,Data_Iteração_Mes,Data_Iteração_Dia,Data_Iteração_DiaSemana,Data_Iteração_Hora,Data_Iteração_FimDeSemana,Data_Iteração_Periodo
0,SIG_TECNICO,-,-,ELIDIONETE DE ANDRADE,sim,- SIG_TECNICO ELIDIONETE DE ANDRADE TATIANA MA...,2017,12,29,Sexta-feira,16,0,Tarde,2018.0,1.0,9.0,Terã§a-feira,11.0,0,Manhã,2018.0,1.0,11.0,Quinta-feira,10.0,0,Manhã,2018.0,1.0,4.0,Quinta-feira,8.0,0,Manhã
1,SIGAA,-,-,ELIDIONETE DE ANDRADE,sim,- SIGAA ELIDIONETE DE ANDRADE TATIANA MAYUMI N...,2017,12,29,Sexta-feira,16,0,Tarde,2018.0,1.0,9.0,Terã§a-feira,11.0,0,Manhã,2018.0,1.0,11.0,Quinta-feira,10.0,0,Manhã,2018.0,1.0,4.0,Quinta-feira,8.0,0,Manhã
2,SIGAA,-,-,-,sim,- SIGAA LARISSA DINIZ RIBEIRO TATIANA MAYUMI N...,2017,12,29,Sexta-feira,14,0,Tarde,2018.0,1.0,2.0,Terã§a-feira,9.0,0,Manhã,2018.0,1.0,4.0,Quinta-feira,8.0,0,Manhã,NaN,NaN,NaN,None,NaN,0,None
3,SIG_TECNICO,-,-,FABIANA APARECIDA PEREIRA DA SILVA,sim,- SIG_TECNICO FABIANA APARECIDA PEREIRA DA SIL...,2017,12,29,Sexta-feira,11,0,Manhã,2018.0,2.0,2.0,Sexta-feira,13.0,0,Tarde,2018.0,2.0,2.0,Sexta-feira,13.0,0,Tarde,2017.0,12.0,29.0,Sexta-feira,13.0,0,Tarde
4,SIGAA,-,-,FABIANA APARECIDA PEREIRA DA SILVA,sim,- SIGAA FABIANA APARECIDA PEREIRA DA SILVA LUC...,2017,12,29,Sexta-feira,11,0,Manhã,2018.0,2.0,2.0,Sexta-feira,13.0,0,Tarde,2018.0,2.0,2.0,Sexta-feira,13.0,0,Tarde,2017.0,12.0,29.0,Sexta-feira,13.0,0,Tarde


In [4]:
# infos
print(df_filtrado.info())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 62810 entries, 0 to 62809
Data columns (total 34 columns):
 #   Column                       Non-Null Count  Dtype  
---  ------                       --------------  -----  
 0   Grupo                        62810 non-null  object 
 1   Categoria                    62810 non-null  object 
 2   Satisfação                   62810 non-null  object 
 3   Usuário Iteração             62810 non-null  object 
 4   Resolvido                    62810 non-null  object 
 5   texto                        62810 non-null  object 
 6   Data_Abertura_Ano            62810 non-null  int32  
 7   Data_Abertura_Mes            62810 non-null  int32  
 8   Data_Abertura_Dia            62810 non-null  int32  
 9   Data_Abertura_DiaSemana      62810 non-null  object 
 10  Data_Abertura_Hora           62810 non-null  int32  
 11  Data_Abertura_FimDeSemana    62810 non-null  int32  
 12  Data_Abertura_Periodo        62810 non-null  object 
 13  Data_Solução_Ano

In [5]:
# Estatisticas basicas 
df_filtrado.describe(include='all')

,Grupo,Categoria,Satisfação,Usuário Iteração,Resolvido,texto,Data_Abertura_Ano,Data_Abertura_Mes,Data_Abertura_Dia,Data_Abertura_DiaSemana,Data_Abertura_Hora,Data_Abertura_FimDeSemana,Data_Abertura_Periodo,Data_Solução_Ano,Data_Solução_Mes,Data_Solução_Dia,Data_Solução_DiaSemana,Data_Solução_Hora,Data_Solução_FimDeSemana,Data_Solução_Periodo,Data_Fechamento_Ano,Data_Fechamento_Mes,Data_Fechamento_Dia,Data_Fechamento_DiaSemana,Data_Fechamento_Hora,Data_Fechamento_FimDeSemana,Data_Fechamento_Periodo,Data_Iteração_Ano,Data_Iteração_Mes,Data_Iteração_Dia,Data_Iteração_DiaSemana,Data_Iteração_Hora,Data_Iteração_FimDeSemana,Data_Iteração_Periodo
count,62810,62810,62810,62810,62810,62810,62810.000000,62810.000000,62810.000000,62810,62810.000000,62810.000000,62810,60243.000000,60243.000000,60243.000000,60243,60243.000000,62810.000000,60243,60188.000000,60188.000000,60188.000000,60188,60188.000000,62810.000000,60188,31528.000000,31528.000000,31528.000000,31528,31528.000000,62810.000000,31528
unique,76,185,7,733,2,62810,NaN,NaN,NaN,7,NaN,NaN,4,NaN,NaN,NaN,7,NaN,NaN,4,NaN,NaN,NaN,7,NaN,NaN,4,NaN,NaN,NaN,7,NaN,NaN,4
top,SIPAC,-,-,-,sim,- SIG_TECNICO ELIDIONETE DE ANDRADE TATIANA MA...,NaN,NaN,NaN,Segunda-feira,NaN,NaN,Tarde,NaN,NaN,NaN,Terã§a-feira,NaN,NaN,Tarde,NaN,NaN,NaN,Quinta-feira,NaN,NaN,Manhã,NaN,NaN,NaN,Terã§a-feira,NaN,NaN,Tarde
freq,12779,34936,61226,31301,60188,1,NaN,NaN,NaN,13561,NaN,NaN,31317,NaN,NaN,NaN,12724,NaN,NaN,31552,NaN,NaN,NaN,12879,NaN,NaN,34605,NaN,NaN,NaN,6637,NaN,NaN,16746
mean,NaN,NaN,NaN,NaN,NaN,NaN,2018.906957,6.074765,15.230791,NaN,12.700907,0.007228,NaN,2018.933918,6.130372,15.473150,NaN,13.064987,0.001544,NaN,2018.932511,6.185967,15.498953,NaN,9.181299,0.170339,NaN,2019.012909,6.019982,15.260055,NaN,12.929491,0.000701,NaN
std,NaN,NaN,NaN,NaN,NaN,NaN,1.693169,3.331826,8.780438,NaN,3.344952,0.084711,NaN,1.693065,3.329343,8.758343,NaN,3.378506,0.039268,NaN,1.691620,3.334437,8.780584,NaN,4.646980,0.375933,NaN,1.766258,3.300530,8.677718,NaN,3.244404,0.026458,NaN
min,NaN,NaN,NaN,NaN,NaN,NaN,2017.000000,1.000000,1.000000,NaN,0.000000,0.000000,NaN,2017.000000,1.000000,1.000000,NaN,0.000000,0.000000,NaN,2017.000000,1.000000,1.000000,NaN,0.000000,0.000000,NaN,2017.000000,1.000000,1.000000,NaN,0.000000,0.000000,NaN
25%,NaN,NaN,NaN,NaN,NaN,NaN,2017.000000,3.000000,7.000000,NaN,10.000000,0.000000,NaN,2018.000000,3.000000,8.000000,NaN,10.000000,0.000000,NaN,2018.000000,3.000000,8.000000,NaN,8.000000,0.000000,NaN,2017.000000,3.000000,8.000000,NaN,10.000000,0.000000,NaN
50%,NaN,NaN,NaN,NaN,NaN,NaN,2019.000000,6.000000,15.000000,NaN,13.000000,0.000000,NaN,2019.000000,6.000000,15.000000,NaN,13.000000,0.000000,NaN,2019.000000,6.000000,15.000000,NaN,9.000000,0.000000,NaN,2019.000000,6.000000,15.000000,NaN,13.000000,0.000000,NaN
75%,NaN,NaN,NaN,NaN,NaN,NaN,2020.000000,9.000000,23.000000,NaN,15.000000,0.000000,NaN,2020.000000,9.000000,23.000000,NaN,16.000000,0.000000,NaN,2020.000000,9.000000,23.000000,NaN,11.000000,0.000000,NaN,2021.000000,9.000000,23.000000,NaN,15.000000,0.000000,NaN


In [6]:
df_total = df_filtrado
#para limitar linhas e testar 
#df_total = df_total.head(10000)
print(f"Tamanho da base limitada: {len(df_total)} linhas")

Tamanho da base limitada: 62810 linhas


In [7]:
df_total.dtypes

Grupo                           object
Categoria                       object
Satisfação                      object
Usuário Iteração                object
Resolvido                       object
texto                           object
Data_Abertura_Ano                int32
Data_Abertura_Mes                int32
Data_Abertura_Dia                int32
Data_Abertura_DiaSemana         object
Data_Abertura_Hora               int32
Data_Abertura_FimDeSemana        int32
Data_Abertura_Periodo           object
Data_Solução_Ano               float64
Data_Solução_Mes               float64
Data_Solução_Dia               float64
Data_Solução_DiaSemana          object
Data_Solução_Hora              float64
Data_Solução_FimDeSemana         int32
Data_Solução_Periodo            object
Data_Fechamento_Ano            float64
Data_Fechamento_Mes            float64
Data_Fechamento_Dia            float64
Data_Fechamento_DiaSemana       object
Data_Fechamento_Hora           float64
Data_Fechamento_FimDeSema

In [8]:


# Callback que atualiza a barra de progresso do Optuna
def tqdm_callback_optuna(study, trial):
    pbar_optuna.update(1)


# Tratar classes raras
min_samples = 2
counts = df_total['Grupo'].value_counts()
valid_classes = counts[counts >= min_samples].index
df_total['Grupo_tratado'] = df_total['Grupo'].apply(lambda x: x if x in valid_classes else "Outros")

# LabelEncoder final
le = LabelEncoder()
df_total['Grupo_encoded'] = le.fit_transform(df_total['Grupo_tratado'])


# Inicializa modelo de embeddings (DistilBERT)

# Inicializa modelo de embeddings (DistilBERT variante via SentenceTransformer)
# Obs: Estamos usando 'distiluse-base-multilingual-cased-v2', que é uma 
# variante do DistilBERT otimizada para gerar embeddings de sentenças em múltiplos idiomas.
# Essa versão não é para classificação direta, mas para extrair vetores semânticos
# que podem ser usados como features em modelos clássicos (ex: XGBoost)
model = SentenceTransformer('distiluse-base-multilingual-cased-v2')

# 
# Gera embeddings

texts = df_total['texto'].astype(str).tolist()
print("Gerando embeddings...")
embeddings = np.array(model.encode(texts, show_progress_bar=True, batch_size=32))


# Divisão treino/teste (80/20) - com stratify

if df_total['Grupo_encoded'].nunique() > 1:
    X_train, X_test, y_train, y_test = train_test_split(
        embeddings,
        df_total['Grupo_encoded'],
        test_size=0.2,
        random_state=42,
        stratify=df_total['Grupo_encoded']
    )


    # Ajusta n_splits automaticamente, baseado na classe mais rara 
    #aqui estava tendo problemas com folds estaticos, existem classes que nao chegam a ter 6 amostras para estratificar
    min_class_count = pd.Series(y_train).value_counts().min()
    n_splits = min(6, min_class_count)  # até 6 folds
    if n_splits < 2:
        print("⚠️ Classes muito raras, não é possível rodar cross-validation")
        df_total['pred_distilbert_XGB'] = None
    else:
        print(f"Usando StratifiedKFold com n_splits={n_splits}")


        # Função objetivo para Optuna
        def objective(trial):
            try:    
                    #OBS: Diferente de outras bibliotecas como LightGBM e CatBoost,
                    # No XGBoost, a métrica de pureza usada para definir o split é o gain.
                    #O parâmetro que define o mínimo ganho exigido para aceitar uma divisão é o gamma.
                params = {
                    'objective': 'multi:softprob',  # classificação multiclasse com probabilidades
                    'num_class': df_total['Grupo_encoded'].nunique(),  # número de classes
                    'eval_metric': 'mlogloss',  # métrica de avaliação
                    'use_label_encoder': False,  # desativa encoder interno
                    'tree_method': 'gpu_hist',  # histograma otimizado em GPU
                    'predictor': 'gpu_predictor',  # predição via GPU

                    # ------------------------------
                    # Controle de aprendizado:
                
                    # Taxa de aprendizado: controla o passo de ajuste a cada árvore.
                    # Valores altos → mais agressivo, maior risco de overfitting.
                    # Valores baixos → mais estável, mas precisa de mais árvores para não underfitting.
                    'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.2),

                    # Número de árvores (boosting rounds)
                    # Valores altos → maior risco de overfitting se learning_rate não for baixo.
                    # Valores baixos → menor risco de overfitting, mas pode causar underfitting.
                    'n_estimators': trial.suggest_int('n_estimators', 30, 60),

                    # ------------------------------
                    # Complexidade da árvore

                    # Profundidade máxima das árvores, controla complexidade, mesmo que haja ganho nos splits limita a profundidade da arvore
                    # Valores altos → árvores mais complexas, maior risco de overfitting.
                    # Valores baixos → árvores mais simples, risco de underfitting.
                    'max_depth': trial.suggest_int('max_depth', 3, 8),

                    # Peso mínimo das instâncias em uma folha, mínimo de hessiano total necessário em um nó filho (controla nós muito pequenos).
                    # Valores baixos → folhas pequenas, maior risco de overfitting.
                    # Valores altos → folhas mais robustas, risco de underfitting.
                    'min_child_weight': trial.suggest_int('min_child_weight', 1, 10),

                    # Penalidade mínima para realizar um split (é tipo o threshold de ganho da divisão, quanto maior significa que o ganho precisa ser muito forte para fazer o split)
                    # Valores altos → menos splits, árvores mais simples, risco de underfitting.
                    # Valores baixos → mais splits, árvores mais complexas, maior risco de overfitting.
                    'gamma': trial.suggest_float('gamma', 0, 2),

                    # ------------------------------
                    # Amostragem e diversidade

                    # Proporção de amostras usadas por árvore
                    # Valores altos → menos diversidade, maior risco de overfitting.
                    # Valores baixos → mais robusto, risco de underfitting.
                    'subsample': trial.suggest_float('subsample', 0.5, 0.8),

                    # Proporção de features usadas em cada árvore
                    # Valores altos → menos diversidade, maior risco de overfitting.
                    # Valores baixos → mais robusto, risco de underfitting.
                    'colsample_bytree': trial.suggest_float('colsample_bytree', 0.5, 0.8),

                    # ------------------------------
                    # Regularização
  
                    # Regularização L1 (reg_alpha): força esparsidade, zerando coeficientes (regularização que aparece no cálculo do ganho.)
                    # Valores altos → mais regularização, menos overfitting, risco de underfitting.
                    # Valores baixos → menos regularização, maior risco de overfitting.
                    'reg_alpha': trial.suggest_float('reg_alpha', 0, 2),

                    # Regularização L2 (reg_lambda): penaliza pesos grandes, suavizando o modelo (regularização que aparece no cálculo do ganho.)
                    # Valores altos → mais regularização, menos overfitting, risco de underfitting.
                    # Valores baixos → menos regularização, maior risco de overfitting.
                    'reg_lambda': trial.suggest_float('reg_lambda', 0, 2),

                    # ------------------------------
                    # Reprodutibilidade

                    'random_state': 42  # garante resultados consistentes
                }




                clf_trial = XGBClassifier(**params)
                skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=42)
                scores = cross_val_score(clf_trial, X_train, y_train, cv=skf, scoring='f1_macro')
                return scores.mean()
            except Exception as e:
                print(f"Erro no trial {trial.number}: {e}")
                return 0.0


        # Executa Optuna

        import optuna
        import logging
        from tqdm import tqdm

        # Desliga logs do Optuna, aqueles de resultados a cada trial, como valores dos hiperparametros
        # entao caso queira ver esses resultados a cada trial, basta comentar a linha abaixo, 
        # ou salvar os resultados em um banco de dados e ver via browser por exemplo (tenho projetos que fazem isso, basta olhar no github)

        optuna.logging.set_verbosity(optuna.logging.WARNING) #so avisos e erros
        #optuna.logging.set_verbosity(optuna.logging.ERROR) #so erros

        n_trials = 50

        pbar_optuna = tqdm(total=n_trials, desc="Busca Hiperparâmetros Optuna", ncols=100)

        def tqdm_callback_optuna(study, trial):
            pbar_optuna.update(1)

        sampler_ = optuna.samplers.TPESampler(
            n_startup_trials=5,
            n_ei_candidates=24,
            seed=42,
            multivariate=True,
            warn_independent_sampling=False
        )

        study = optuna.create_study(direction="maximize", sampler=sampler_)
        study.optimize(objective, n_trials=n_trials, callbacks=[tqdm_callback_optuna])

        pbar_optuna.close()


        if len(study.trials) == 0 or study.best_trial is None:
            print("⚠️ Nenhum trial completado com sucesso.")
            df_total['pred_distilbert_XGB'] = None
        else:
            print("Melhores hiperparâmetros encontrados:")
            for param, valor in study.best_params.items():
                print(f"  {param}: {valor}")


            # Treino final com os melhores hiperparâmetros

            best_params = study.best_params
            best_params.update({
                'objective': 'multi:softprob',
                'num_class': df_total['Grupo_encoded'].nunique(),
                'eval_metric': 'mlogloss',
                'use_label_encoder': False,
                'tree_method': 'gpu_hist',
                'predictor': 'gpu_predictor',
                'random_state': 42
            })

            clf = XGBClassifier(**best_params)

            # Barra de progresso do treino
            n_estimators = best_params['n_estimators']
            pbar_train = tqdm(total=n_estimators, desc="Treino XGBoost", ncols=100)

            # Treina iterativamente usando warm_start para atualizar o modelo
            clf.set_params(n_estimators=1, warm_start=True)  # inicializa com 1 árvore
            for i in range(n_estimators):
                clf.set_params(n_estimators=i+1)  # adiciona uma árvore por vez
                clf.fit(embeddings, df_total['Grupo_encoded'], verbose=False)
                pbar_train.update(1)

            pbar_train.close()

            # ------------------------------
            # Predição completa

            preds_enc = clf.predict(embeddings)
            preds_proba = clf.predict_proba(embeddings)
            df_total['pred_distilbert_XGB'] = preds_enc

            # ------------------------------
            # Métricas de avaliação

            y_true = df_total['Grupo_encoded']
            accuracy_xgb = accuracy_score(y_true, preds_enc)
            f1_macro_xgb = f1_score(y_true, preds_enc, average='macro')
            f1_weighted_xgb = f1_score(y_true, preds_enc, average='weighted')
            print(f"Accuracy (XGB): {accuracy_xgb:.4f}")
            print(f"F1 Macro (XGB): {f1_macro_xgb:.4f}")
            print(f"F1 Weighted (XGB): {f1_weighted_xgb:.4f}")

            y_bin = label_binarize(y_true, classes=np.unique(y_true))
            if y_bin.shape[1] > 1:
                auc_xgb = roc_auc_score(y_bin, preds_proba, average='macro')
                print(f"AUC Macro (XGB): {auc_xgb:.4f}")
            else:
                print("AUC não calculada: menos de 2 classes presentes.")

            logloss_xgb = log_loss(y_true, preds_proba)
            print(f"Log Loss (XGB): {logloss_xgb:.4f}")

            brier_scores = [brier_score_loss(y_bin[:, i], preds_proba[:, i]) for i in range(y_bin.shape[1])]
            print(f"Brier Score Macro (XGB): {np.mean(brier_scores):.4f}")

else:
    print("Não há classes suficientes para treino.")
    df_total['pred_distilbert_XGB'] = None


Gerando embeddings...


Batches:   0%|          | 0/1963 [00:00<?, ?it/s]

Usando StratifiedKFold com n_splits=2


Busca Hiperparâmetros Optuna: 100%|███████████████████████████████| 50/50 [1:01:28<00:00, 73.77s/it]


Melhores hiperparâmetros encontrados:
  learning_rate: 0.19887602008957977
  n_estimators: 60
  max_depth: 5
  min_child_weight: 2
  gamma: 0.15320247287346014
  subsample: 0.7206682920174232
  colsample_bytree: 0.6182791776562456
  reg_alpha: 0.01656691535564231
  reg_lambda: 0.9249356318260179


Treino XGBoost: 100%|███████████████████████████████████████████████| 60/60 [38:10<00:00, 38.18s/it]


Accuracy (XGB): 0.9998
F1 Macro (XGB): 0.9681
F1 Weighted (XGB): 0.9997
AUC Macro (XGB): 1.0000
Log Loss (XGB): 0.0080
Brier Score Macro (XGB): 0.0000


In [9]:
# Visualiza resultados
print(df_total['pred_distilbert_XGB'].value_counts(dropna=False))


pred_distilbert_XGB
61    12785
50     7998
72     7821
54     5697
9      3760
38     3361
28     2428
57     2272
29     2001
0      1538
13     1352
68     1296
59     1255
20      969
65      892
26      874
30      828
70      702
10      590
4       423
51      416
66      342
62      335
35      263
21      248
60      176
32      169
56      154
52      145
55      141
74      139
1       133
12      129
22      125
40      114
71      102
44       89
69       87
41       64
31       62
48       53
3        50
53       48
18       46
46       41
58       34
64       30
15       29
42       22
19       19
24       18
67       15
23       14
49       13
47       12
8        10
11        8
73        8
25        7
36        6
33        6
45        6
37        5
63        5
5         5
6         4
7         4
2         3
43        3
34        3
39        3
14        3
27        2
Name: count, dtype: int64


In [10]:
df_total.head(5)

,Grupo,Categoria,Satisfação,Usuário Iteração,Resolvido,texto,Data_Abertura_Ano,Data_Abertura_Mes,Data_Abertura_Dia,Data_Abertura_DiaSemana,Data_Abertura_Hora,Data_Abertura_FimDeSemana,Data_Abertura_Periodo,Data_Solução_Ano,Data_Solução_Mes,Data_Solução_Dia,Data_Solução_DiaSemana,Data_Solução_Hora,Data_Solução_FimDeSemana,Data_Solução_Periodo,Data_Fechamento_Ano,Data_Fechamento_Mes,Data_Fechamento_Dia,Data_Fechamento_DiaSemana,Data_Fechamento_Hora,Data_Fechamento_FimDeSemana,Data_Fechamento_Periodo,Data_Iteração_Ano,Data_Iteração_Mes,Data_Iteração_Dia,Data_Iteração_DiaSemana,Data_Iteração_Hora,Data_Iteração_FimDeSemana,Data_Iteração_Periodo,Grupo_tratado,Grupo_encoded,pred_distilbert_XGB
0,SIG_TECNICO,-,-,ELIDIONETE DE ANDRADE,sim,- SIG_TECNICO ELIDIONETE DE ANDRADE TATIANA MA...,2017,12,29,Sexta-feira,16,0,Tarde,2018.0,1.0,9.0,Terã§a-feira,11.0,0,Manhã,2018.0,1.0,11.0,Quinta-feira,10.0,0,Manhã,2018.0,1.0,4.0,Quinta-feira,8.0,0,Manhã,SIG_TECNICO,60,60
1,SIGAA,-,-,ELIDIONETE DE ANDRADE,sim,- SIGAA ELIDIONETE DE ANDRADE TATIANA MAYUMI N...,2017,12,29,Sexta-feira,16,0,Tarde,2018.0,1.0,9.0,Terã§a-feira,11.0,0,Manhã,2018.0,1.0,11.0,Quinta-feira,10.0,0,Manhã,2018.0,1.0,4.0,Quinta-feira,8.0,0,Manhã,SIGAA,54,54
2,SIGAA,-,-,-,sim,- SIGAA LARISSA DINIZ RIBEIRO TATIANA MAYUMI N...,2017,12,29,Sexta-feira,14,0,Tarde,2018.0,1.0,2.0,Terã§a-feira,9.0,0,Manhã,2018.0,1.0,4.0,Quinta-feira,8.0,0,Manhã,NaN,NaN,NaN,None,NaN,0,None,SIGAA,54,54
3,SIG_TECNICO,-,-,FABIANA APARECIDA PEREIRA DA SILVA,sim,- SIG_TECNICO FABIANA APARECIDA PEREIRA DA SIL...,2017,12,29,Sexta-feira,11,0,Manhã,2018.0,2.0,2.0,Sexta-feira,13.0,0,Tarde,2018.0,2.0,2.0,Sexta-feira,13.0,0,Tarde,2017.0,12.0,29.0,Sexta-feira,13.0,0,Tarde,SIG_TECNICO,60,60
4,SIGAA,-,-,FABIANA APARECIDA PEREIRA DA SILVA,sim,- SIGAA FABIANA APARECIDA PEREIRA DA SILVA LUC...,2017,12,29,Sexta-feira,11,0,Manhã,2018.0,2.0,2.0,Sexta-feira,13.0,0,Tarde,2018.0,2.0,2.0,Sexta-feira,13.0,0,Tarde,2017.0,12.0,29.0,Sexta-feira,13.0,0,Tarde,SIGAA,54,54
